# Use Case: Conjugate Gradient

In this lab, we extend our 'numerical solver' implemented with matrix-free Jacobi iterations using the conjugate gradient method.
Being familiar with the algorithm on a deeper level is not necessary, but in case you are interested have a look at, e.g., this [wikipedia article](https://en.wikipedia.org/wiki/Conjugate_gradient_method#The_resulting_algorithm).
The linked page also shows an outline of the algorithm implemented which builds on the following building blocks:
* matrix-vector products (i.e. stencil applications)
* other vector operations such as scaling and addition (i.e. similar to the stream pattern)
* vector dot products (i.e. reductions)

Since this algorithm includes multiple steps, we first augment our baseline implementation with markers to make subsequent performance analysis easier.
Code annotation is done with NVTX, as introduced in the [Application Level Profiling](./03-application-level-profiling.ipynb) notebook.

The course material includes a CPU serial base version as well as GPU-accelerated versions based on CUDA, OpenMP and OpenACC:
* [cg-base.cpp](../src/cg/cg-base.cpp),
* [cg-cuda-mm.cu](../src/cg/cg-cuda-mm.cu),
* [cg-omp-target-mm.cpp](../src/cg/cg-omp-target-mm.cpp), and
* [cg-openacc-mm.cpp](../src/cg/cg-openacc-mm.cpp).

As before, parameterization via command line arguments is possible:
- **Data type**: `float` or `double`
- **nx, ny**: Grid dimensions, scaling the total workload (`nx * ny`)
- **nWarmUp**: Number of non-timed warm-up iterations
- **nIt**: Number of timed iterations

Compilation, execution and profiling with Nsight Systems can be done with the below cells.
The produced report files are stored in the `../profiles` folder and are linked below each profiling cell.
You can directly download them by clicking the link with the middle mouse button, or by using shift + right click and then selecting download.

### Base

In [ ]:
!g++ -O3 -march=native -std=c++17 -I/opt/nvidia/hpc_sdk/Linux_x86_64/26.1/cuda/include ../src/cg/cg-base.cpp -o ../build/cg-base

In [ ]:
!../build/cg-base double 8192 8192 2 16

In [ ]:
!nsys profile --stats=true -o ../profiles/cg-base --force-overwrite=true ../build/cg-base double 8192 8192 2 16

Download the [cg-base](../profiles/cg-base.nsys-rep) report file and open it with your local installation of Nsight Systems.

### CUDA

As before, replace `sm_86` in the below cell to match the currently used GPU if it differs from the course's A40.

In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_86 -o ../build/cg-cuda-mm ../src/cg/cg-cuda-mm.cu

In [ ]:
!../build/cg-cuda-mm double 8192 8192 2 16

In [ ]:
!nsys profile --stats=true -o ../profiles/cg-cuda-mm --force-overwrite=true ../build/cg-cuda-mm double 8192 8192 2 16

Download the [cg-cuda-mm](../profiles/cg-cuda-mm.nsys-rep) report file and open it with your local installation of Nsight Systems.

### OpenMP

This code uses **managed memory** which is enabled by using the `-gpu=mem:managed` flag.
This allows a single allocation to be accessed from both CPU and GPU.
The CUDA runtime automatically migrates data pages between host and device as needed.
While this reduces implementation effort, it can also have negative impacts on performance.

In [ ]:
!nvc++ -O3 -std=c++17 -mp=gpu -target=gpu -gpu=mem:managed -o ../build/cg-omp-target-mm ../src/cg/cg-omp-target-mm.cpp

In [ ]:
!../build/cg-omp-target-mm double 8192 8192 2 16

In [ ]:
!nsys profile --stats=true -o ../profiles/cg-omp-target-mm --force-overwrite=true ../build/cg-omp-target-mm double 8192 8192 2 16

Download the [cg-omp-target-mm](../profiles/cg-omp-target-mm.nsys-rep) report file and open it with your local installation of Nsight Systems.

### OpenACC

This code uses **managed memory** which is enabled by using the `-gpu=mem:managed` flag.
This allows a single allocation to be accessed from both CPU and GPU.
The CUDA runtime automatically migrates data pages between host and device as needed.
While this reduces implementation effort, it can also have negative impacts on performance.

In [ ]:
!nvc++ -O3 -std=c++17 -acc=gpu -target=gpu -gpu=mem:managed -o ../build/cg-openacc-mm ../src/cg/cg-openacc-mm.cpp

In [ ]:
!../build/cg-openacc-mm double 8192 8192 2 16

In [ ]:
!nsys profile --stats=true -o ../profiles/cg-openacc-mm --force-overwrite=true ../build/cg-openacc-mm double 8192 8192 2 16

Download the [cg-openacc-mm](../profiles/cg-openacc-mm.nsys-rep) report file and open it with your local installation of Nsight Systems.

## Exercise - Performance Evaluation and Optimization

This exercise is designed to be longer and to give you more flexibility in which techniques you want to experiment with.
The baseline implementations are already partly GPU accelerated, but lack the desired performance.
Your tasks are as follows:
* Review the code(s) and choose one version (or create an independent one).
* Profile the application and check the Nsight GUI command line output and timeline visualization for NVTX data.
* Do some POD iterations.
  * **P**rofile: use Nsight Systems and Compute to isolate hot-spots and performance issues in the application.
  * **O**ptimize: implement performance optimizations to address bottlenecks.
  * **D**eploy: check whether the results are still correct.
* Add your performance result to the leaderboard.

Note that each GPU-accelerated version includes *performance bugs*.
Apart from fixing them, here are some additional optimization ideas to get you started:
* Start with an Nsight Systems analysis and optimize memory transfers based on its results.
* Identify hot spot kernels and optimize them iteratively:
  * Optimize occupancy/ execution configurations if applicable
  * Identify sub-optimal access patterns if applicable
* Perform reductions on GPU
  * \[CUDA\]: use optimized reductions e.g. using CUB or thrust
* Apply kernel fusion
  * \[CUDA\]: apply additional kernel fusion using grid synchronization with cooperative groups
* Add alternating forwards-backwards kernels

## Next Step

Congratulations on finishing this course!

If you want to dive deeper, here are some topics this course did not cover:
* [NVIDIA CUDA Profiling Tools Interface (CUPTI)](https://developer.nvidia.com/cupti) provides means to profile applications programmatically.
* [AMD Tools](https://github.com/ROCm/rocm-systems)
* AMD GPU Profiling Tools (ROCm)
  * Legacy/ superseded
    * `ROCTracer`
    * `ROCProfiler` (v1/ v2)
    * `rocprof` (legacy CLI, superseded by rocprofv3)
    * `rocprofv2`
  * Phase-out
    * `Omniperf` (kernel-level performance analysis)
    * `Omnitrace` -> evolving into `ROCm Systems Profiler`
  * Current core stack
    * `ROCProfiler SDK`
    * `rocprofv3`
  * Current tools
    * [ROCm Systems Profiler](https://rocm.docs.amd.com/projects/rocprofiler-systems/en/latest/index.html) `rocm-systems`
    * [ROCm Compute Profiler](https://rocm.docs.amd.com/projects/rocprofiler-compute/en/latest/index.html) `rocm-compute`


Here are some pointers if you want to further extend your GPU and HPC knowledge:
* [NHR@FAU](https://nhr.fau.de) offers a number of courses on different HPC related topics
  * [https://hpc.fau.de/teaching/tutorials-and-courses/](https://hpc.fau.de/teaching/tutorials-and-courses/)
* Likewise, most compute centers offer a variety of different courses, many of them online and free of charge
* Nvidia's [On-Demand Video Collection](https://www.nvidia.com/en-us/on-demand/) contains thousands of recordings of many insightful talks covering various GPU-related topics.
* [GTC](https://www.nvidia.com/gtc/) is one of the premier conferences around GPU computing and virtual attendance is usually free of charge.
